# Companion Lab 05 — Clinical Fairness, Calibration, and Abstention Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/curriculum-v2/notebooks/05-fairness-and-abstention-audit.ipynb)

Paired with **Chapter 5.3**, this CPU-friendly notebook brings together every evaluation tool from Parts 2 and 5 into a single reusable **Clinical Model Audit Suite**:
1. ROC & Precision–Recall curves + **95% bootstrap confidence intervals**
2. **Calibration (reliability) diagrams** comparing well-calibrated vs. overconfident models
3. **Disaggregated subgroup sensitivity & false-negative gaps** on real clinical diabetes data (`load_diabetes`)
4. **Selective prediction (abstention)** with **per-group coverage checks**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, confusion_matrix

BLUE, VERMILION, TEAL = '#0072B2', '#D55E00', '#009E73'

# Load real clinical diabetes cohort in native clinical units
df = load_diabetes(as_frame=True, scaled=False).frame
X = df.drop(columns='target')
y = (df['target'] >= 180).astype(int)   # 1 = high 1-year progression risk

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)
print(f'Train: {len(X_tr)} | Test: {len(X_te)} | High-risk prevalence: {y.mean():.1%}')

In [ ]:
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=5, random_state=0),
}

rng = np.random.default_rng(0)
def boot_auroc_ci(y_true, probs, n_boot=1000):
    vals = []
    yt = np.asarray(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[idx])) == 2:
            vals.append(roc_auc_score(yt[idx], probs[idx]))
    return np.percentile(vals, [2.5, 97.5])

fitted_probs = {}
for name, clf in models.items():
    clf.fit(X_tr, y_tr)
    p = clf.predict_proba(X_te)[:, 1]
    fitted_probs[name] = p
    lo, hi = boot_auroc_ci(y_te, p)
    print(f'{name:<20} | AUROC = {roc_auc_score(y_te, p):.3f} (95% CI [{lo:.3f}, {hi:.3f}]) | '
          f'AUPRC = {average_precision_score(y_te, p):.3f}')

## 1. Disaggregated Subgroup Audit & Abstention Coverage Equity

In [ ]:
probs = fitted_probs['Logistic Regression']
preds = (probs >= 0.35).astype(int)          # screening threshold
conf  = np.maximum(probs, 1 - probs)

subgroups = {
    'Age < 50':  X_te['age'] < 50,
    'Age >= 50': X_te['age'] >= 50,
    'BMI < 26':  X_te['bmi'] < 26,
    'BMI >= 26': X_te['bmi'] >= 26,
}

print(f'{"Subgroup":<12} | {"n":>3} | {"Sens @0.35":>10} | {"Auto-cov @0.75":>14} | {"Auto-acc":>9}')
print('-' * 60)
for label, mask in subgroups.items():
    m = np.asarray(mask)
    sens = recall_score(y_te[m], preds[m])
    keep = conf[m] >= 0.75
    auto_acc = ((probs[m][keep] >= 0.5).astype(int) == y_te[m][keep]).mean() if keep.sum() else np.nan
    print(f'{label:<12} | {m.sum():>3} | {sens:10.1%} | {keep.mean():14.1%} | {auto_acc:9.1%}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))

# Panel A: Calibration curves
for (name, p), col in zip(fitted_probs.items(), [BLUE, VERMILION]):
    frac, mean_p = calibration_curve(y_te, p, n_bins=6, strategy='quantile')
    ax1.plot(mean_p, frac, 'o-', lw=2.2, color=col, label=name)
ax1.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
ax1.set_xlabel('Predicted risk'); ax1.set_ylabel('Observed high-risk rate')
ax1.set_title('A. Calibration audit'); ax1.legend(frameon=False); ax1.grid(alpha=0.25)

# Panel B: Abstention trade-off
ts = np.linspace(0.50, 0.92, 30)
covs, accs = [], []
for t in ts:
    k = conf >= t
    covs.append(k.mean())
    accs.append(((probs[k] >= 0.5).astype(int) == y_te[k]).mean() if k.sum() else np.nan)
ax2.plot(ts * 100, np.array(covs) * 100, lw=2.2, color=BLUE, label='Coverage (%)')
ax2.plot(ts * 100, np.array(accs) * 100, lw=2.4, color=TEAL, label='Auto-accuracy (%)')
ax2.set_xlabel('Confidence threshold (%)'); ax2.set_ylabel('Percent (%)')
ax2.set_title('B. Selective prediction trade-off'); ax2.legend(frameon=False); ax2.grid(alpha=0.25)
plt.tight_layout()
plt.show()